# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marrwan1/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Method choice: Random Forest Classifier

Why:
- My features (impressions, CTR, position) have non-linear relationships
  with the label — a tree-based model captures this better than logistic regression.
- Random Forest reduces overfitting vs a single Decision Tree.
- It gives permutation importance scores that explain which signals matter most.
- The baseline (W04) used a hand-written rule — Random Forest is the natural
  next step: same features, learned weights instead of fixed thresholds.

Metric: ROC-AUC (same as W04 honest AUC) so the comparison is apples-to-apples.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Split design: time-aware split
- Train : month=2026-03 (features) → month=2026-04 (label)
- Test  : month=2026-04 (features) → month=2026-05 (label)
- No random shuffle — past predicts future, not the reverse.

In [1]:
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

def get_features(month):
    return con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)  AS gsc_impressions,
        SUM(gsc_clicks)       AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        CASE WHEN SUM(gsc_impressions) = 0 THEN NULL
             ELSE SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) END AS ctr,
        COUNT(DISTINCT report_date) AS days_with_data,
        SUM(gsc_clicks) AS total_clicks
    FROM read_parquet('{BASE}/month={month}/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
    """).df()

def get_label(month):
    return con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS total_clicks_next
    FROM read_parquet('{BASE}/month={month}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    """).df()

# Train
train_feat  = get_features("2026-03")
train_label = get_label("2026-04")
train = train_feat.merge(train_label, on=["client_hash_id","content_hash_id"], how="inner").dropna()
train["label"] = (train["total_clicks_next"] > train["total_clicks"] * 1.20).astype(int)

# Test
test_feat  = get_features("2026-04")
test_label = get_label("2026-05")
test = test_feat.merge(test_label, on=["client_hash_id","content_hash_id"], how="inner").dropna()
test["label"] = (test["total_clicks_next"] > test["total_clicks"] * 1.20).astype(int)

print(f"Train: {len(train):,} rows | Positive: {train['label'].mean():.1%}")
print(f"Test : {len(test):,}  rows | Positive: {test['label'].mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train: 158,549 rows | Positive: 16.2%
Test : 183,345  rows | Positive: 20.3%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
FEATURES = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr", "days_with_data"]

X_train, y_train = train[FEATURES], train["label"]
X_test,  y_test  = test[FEATURES],  test["label"]

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

# Baseline (W04 rule reproduced on test set)
test["expected_ctr"] = test["gsc_avg_position"].apply(
    lambda p: 0.0048 if p <= 3 else 0.0035 if p <= 10 else 0.0028 if p <= 20 else 0.0013
)
test["baseline_score"] = (test["expected_ctr"] - test["ctr"]) * test["gsc_impressions"]
baseline_auc = roc_auc_score(y_test, test["baseline_score"])

print("Model vs Baseline")
print("=" * 35)
print(f"W04 Baseline (rule)  AUC: {baseline_auc:.4f}")
print(f"W05 Random Forest    AUC: {rf_auc:.4f}")
print(f"Improvement          : {rf_auc - baseline_auc:+.4f}")

Model vs Baseline
W04 Baseline (rule)  AUC: 0.6364
W05 Random Forest    AUC: 0.7697
Improvement          : +0.1333


Interpretation:

Feature importance:
- gsc_impressions and days_with_data are the strongest signals — the model
  relies on volume and consistency, not CTR or position directly.
- ctr and gsc_avg_position have near-zero importance — the hand-written
  rule (W04) leaned on these heavily, which explains the lower baseline AUC.

Errors:
- False Positives (292): low-impression pages at position ~13 with decent
  CTR — the model flags them but they don't grow. Likely pages with no
  refresh potential, just stable low-traffic content.

- False Negatives (35,595): high-impression pages (avg 2,429) with very
  low CTR (0.002) and high days_with_data (25) — the model misses them
  because their profile looks like stale content, not growth candidates.
  These are the most costly errors — missed real opportunities.

Main weakness: the model is conservative — it misses too many true
growth pages (high FN). A lower decision threshold or a recall-focused
metric would help in Week 6.

In [4]:
import pandas as pd

# Permutation importance
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature"   : FEATURES,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print("Permutation Importance:")
print(imp_df.to_string(index=False))

# Error analysis
test["rf_score"] = rf.predict_proba(X_test)[:, 1]
test["pred"]     = (test["rf_score"] >= 0.5).astype(int)

FP = test[(test["pred"] == 1) & (y_test.values == 0)]
FN = test[(test["pred"] == 0) & (y_test.values == 1)]

print(f"\nFalse Positives (flagged but didn't grow): {len(FP):,}")
print(f"False Negatives (missed growth)          : {len(FN):,}")

print("\nFP avg profile:")
print(FP[FEATURES].mean().round(4))

print("\nFN avg profile:")
print(FN[FEATURES].mean().round(4))

Permutation Importance:
         feature  importance
 gsc_impressions    0.071376
  days_with_data    0.051462
      gsc_clicks    0.001726
gsc_avg_position    0.000425
             ctr    0.000191

False Positives (flagged but didn't grow): 292
False Negatives (missed growth)          : 35,595

FP avg profile:
gsc_impressions     780.3562
gsc_clicks            9.4589
gsc_avg_position     12.7251
ctr                   0.0104
days_with_data        8.7089
dtype: float64

FN avg profile:
gsc_impressions     2429.0596
gsc_clicks             5.2060
gsc_avg_position      13.7407
ctr                    0.0023
days_with_data        25.0901
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.